<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


# ROS 2 Nodes, Topics, and Interfaces

A camera image and a pose estimate both carry information, but a controller cannot use them interchangeably. Even two messages containing a field called `data` may represent different things.

For separately developed nodes to cooperate, their interfaces must agree on how information is exchanged and what it means. This notebook examines that agreement, then compares the ROS 2 interfaces available for streams, requests, longer-running tasks, and configuration.

## Nodes

A node's responsibility describes the work it performs. Its publishers, subscriptions, and other interfaces describe how that work connects to the rest of the application.

In the camera-to-controller example, the pose-estimation node subscribes to images and publishes pose estimates. Publishing and subscribing are roles within a node; they do not require separate nodes.

A node is also distinct from the process that hosts it. One running program can contain several nodes, each with its own name and responsibility. The [ROS 2 node model](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Nodes.html) supports communication both within a process and between processes.

<!--
Proposed checkpoint replacement
ID: node-definition
Source: #nodes
Question: A single running program contains a camera node and a pose-estimation node. Does sharing a process make them one ROS 2 node? Explain.
Model answer: No. A process can host several named ROS 2 nodes. The camera and pose-estimation nodes retain their separate responsibilities and graph interfaces even when they share a process.
-->

## Topics and messages

A topic identifies a stream of messages. The topic name identifies the stream, while the message type describes the structure of each message.

For example, the local publisher used later in this LX publishes on `chatter` using `std_msgs/msg/String`. Here, `chatter` is the topic name, and `std_msgs/msg/String` is the message type. Other topics can use that same type.

A topic can have multiple publishers and subscribers. A publisher sends messages without naming each receiving node, allowing another subscriber to consume the same stream without changing the publisher's implementation.

## Message contracts

A shared topic name alone is insufficient for communication. The publisher and subscriber also need compatible message types and delivery settings.

The meaning of the fields matters as well. If one component interprets a distance in metres and another interprets it in centimetres, the exchanged numbers can have the correct field type while producing an incorrect result. The interface agreement therefore includes both the message structure and its documented meaning.

__Quality of service (QoS)__ describes delivery policies, such as whether lost messages should be retransmitted and how much recent message history is retained. Publisher and subscriber policies must be compatible. Later graph-inspection activities will make these settings visible.

### Compare two message definitions

In the prepared ROS 2 exercise terminal, the following commands display two installed message definitions:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
ros2 interface show std_msgs/msg/String
ros2 interface show std_msgs/msg/Int32
```

Both definitions contain a field named `data`, but its type differs. The relevant fields are summarized below; the full output also contains comments.

| Message type | Field definition | Example meaning |
|---|---|---|
| [`std_msgs/msg/String`](https://github.com/ros2/common_interfaces/blob/jazzy/std_msgs/msg/String.msg) | `string data` | Text, such as `"42"` |
| [`std_msgs/msg/Int32`](https://github.com/ros2/common_interfaces/blob/jazzy/std_msgs/msg/Int32.msg) | `int32 data` | A whole number, such as `42` |

A subscriber expecting `Int32` cannot receive a `String` message merely because both fields are named `data`. Text containing digits is still text; ROS 2 does not automatically convert it into an integer for the subscriber.

These simple types make the difference easy to inspect. A sensor-specific interface can provide more meaning through its fields and documentation, such as the units associated with an acceleration measurement.

<!--
Proposed checkpoint replacement
ID: message-contract
Source: #message-contracts
Question: A publisher sends std_msgs/msg/String on chatter with data containing "42". A subscriber expects std_msgs/msg/Int32 on chatter. Is the shared topic name and data field enough for communication? Why?
Model answer: No. The message types differ: one carries text and the other carries a signed 32-bit integer. The shared field name does not make those interfaces compatible, and ROS 2 does not automatically convert the text into an integer.
-->

### Compare installed definitions with the visible graph

The next commands inspect nodes and topics visible from the terminal:

```bash
ros2 node list
ros2 topic list -t
```

`ros2 node list` reports discovered node names, while `ros2 topic list -t` reports discovered topic names together with their types. Notice which nodes and topics, if any, are visible in your local graph; and consider that an empty list is normal before starting a node.

These results describe the graph visible to this terminal. By contrast, `ros2 interface show` reads an installed definition. A message definition can therefore be available even when no node using it appears in the graph.

This distinction helps interpret an apparently empty result: successful definition lookup and successful discovery answer different questions. A later activity will start a publisher so that its node, topic, and messages can be inspected together.

## Services, actions, and parameters

Topics suit streams such as camera images or pose estimates, but other than exchanging data, nodes can have other interactions. For example:

* A [service](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Services.html) provides a request and a response. It suits a short operation, such as requesting the current status of a component.

* An [action](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Actions.html) represents work toward a goal, with progress feedback, a final result, and support for cancellation. For example, a navigation task may take time, report progress, and need to be cancelled before completion.

* A [parameter](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Parameters.html) is a named configuration value belonging to a node. A node might expose a controller gain or a publication rate as a parameter. The node must implement that interface; a constant in a Python file does not automatically become a ROS 2 parameter unless programmed to be one.

The following examples illustrate interface choices rather than interfaces guaranteed to exist on a particular Duckiedrone.

| Need | Suitable interface | Reason |
|---|---|---|
| Receive successive camera images | Topic | Each new image contributes to a stream |
| Request the current status of a component | Service | One request produces a response |
| Navigate to a goal while reporting progress | Action | The task has a goal, feedback, completion, and cancellation |
| Configure a node's controller gain | Parameter | The value configures that node's behavior |

The choice follows the interaction required. A long-running task that needs progress feedback and cancellation requires more than a single request-response exchange.

<!--
Proposed checkpoint replacement
ID: interface-selection
Source: #services-actions-and-parameters
Question: A navigation task must report progress and support cancellation before reaching its goal. Which interface best expresses those requirements?
Choices:
A. A topic carrying only the latest position.
B. A service returning a response when navigation finishes.
C. An action accepting a goal and providing feedback, a result, and cancellation.
Correct choice: C.
Model answer: An action provides the required task structure. A position topic can report observations but does not by itself define the goal and its cancellation. A service provides a request-response exchange without the action's progress and cancellation interface.
-->

## Further reading

The ROS 2 Jazzy documentation describes [topics](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Topics.html) and [QoS settings](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Quality-of-Service-Settings.html) in more detail.

## Checkpoint

The next cell contains the interactive self-check. A response must be entered or selected before the corresponding answer can be revealed.

## Summary and next steps

A node can combine several communication roles. Topics identify message streams, while message types and documented meaning define how their contents should be interpreted. Services, actions, and parameters support requests, goal-directed tasks, and configuration.

Compatible interfaces still need a way to find one another. [Notebook 4: ROS 2 Discovery, Domains, and Names](./4-ros-2-discovery-domains-and-names.ipynb) examines how discovery settings and name resolution affect the graph visible from a terminal.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
